# 29 · 外部验证：换一批别人整理的病人

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → 28 时间外验证 ✅ → [29 外部验证 ✅]
```

> **这一章的代码在分支 `feat/temporal-external` 里**（和第 28 章同一个合并请求）。两组外部病人共 712 道题；DeepSeek 约 ¥30，OpenRouter 约 $6.5。**运行这本笔记不花钱**：考试的命令只列出来不重跑，结果从 MLflow 和逐题表里读。
>
> **这一章也分两段写。** 第 1–4 步是考之前写的，第 5 步起是考完以后填的。

## 背景：题没重复，但都是同一位老师出的

到第 28 章为止，所有考题都来自同一个地方：**Phenopacket Store**。留出集和开发集按论文分开了，题没有重复，但是：

- **同一批人、按同一套习惯整理**：哪种表现记成哪个 HPO 编号、记多细，都是一个团队的风格。"找相似病人"的病例库也出自这个团队，所以它在我们的题上可能比在真实世界里更准（[`docs/leakage.md`](../leakage.md) 第六节第 5 条早就写了这个担心）
- **都是发表过的病例报告**：论文里写的多是"这个病的典型病人"，真实门诊的病人常常不典型

**打个比方**：你一直做同一位老师出的题，题目没重复，但出题风格一样，你可能只是摸透了这位老师。换一位老师出题，才知道是不是真会。

**公司里的做法**：这叫**外部验证**（external validation）：换一个来源、换一批人整理的数据再考一次。和它相对，前面做的都是**内部验证**。医疗 AI 产品申请上市时，审评几乎一定会问"有没有在别的医院的数据上验证过"；临床预测模型的报告规范 TRIPOD 也把"内部验证"和"外部验证"分成不同的证据等级。

## 这一章做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 找外部数据 | RareBench，固定在 HuggingFace 的一次提交 | 别人整理的病人 |
| 2 | 查重叠：哪几组真算"外部" | 逐个比对症状 | 防止"外部"数据其实是同一批病人 |
| 3 | 读进来：诊断编号怎么认 | 多个编号、Orphanet 精确对应、没有论文的病人怎么考 | 判分规则事先说清 |
| 4 | 选题、先写方案 | 预注册：`docs/external_plan.md`；先算"地板" | 不看了结果再挑；分数和什么比 |
| 5 | 开考 | 每组 9–10 次考试，`--final`，泄露审计 | 只考一次 |
| 6 | 结果 | 置信区间、逐题配对、两种判法 | 回答方案里的问题 |
| 7 | 和内部验证比，掉了多少 | 同一个系统，两份题的分数并排 | 看出谁依赖 Phenopacket Store 的写法 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 找外部数据：RareBench

### 在干什么
找一批**不是 Phenopacket Store、也不是 HPO 整理资料时引用的论文**里来的病人。

### 背景
罕见病诊断领域，大家用来互相比较的公开数据不多。最常用的是 **RareBench**（清华大学等，KDD 2024），DeepRare 的论文也用它。它公开了四组病人：

| 组 | 人数 | 来源 |
|---|---|---|
| HMS | 88 | 德国汉诺威医学院一个门诊的病人（2019 年一项研究附带公开） |
| RAMEDIS | 624 | 医生提交病例的代谢病数据库（2010 年发表） |
| MME | 40 | 已发表的病例报告（Matchmaker Exchange 的测试集） |
| LIRICAL | 370 | 已发表的病例报告（LIRICAL 论文自己的测试集） |

第五组是北京协和医院（PUMCH）的病人：说明里写公开了其中 75 例，但 HuggingFace 上的数据包里没有这一组，所以用不了。

每个病人只有两样东西：**观察到的 HPO 编号**和**诊断编号**。没有性别，没有基因，没有出处论文。

### 用了什么技术，为什么
- **固定在 HuggingFace 的一次提交**（`6f054e0`），登记在 `evals/datasets.toml`，带校验码。和其他数据一个规矩
- 数据包本身就是几个 JSON 行文件，直接读，不需要它附带的加载脚本（新版 `datasets` 已经不再支持这种脚本）

### 公司里什么情况怎么做
- **用公开评测集之前，先读它的说明和原论文**：数据从哪来、怎么整理的、有没有和自己的训练数据重叠。别人用了，不等于它适合你的问题
- **外部数据和内部数据一样要固定版本**：HuggingFace 上的数据集也会更新

### 执行

In [ ]:
run("uv run raredx data fetch rarebench")
import json, zipfile
archive = zipfile.ZipFile(ROOT / "data/raw/rarebench/6f054e0/data.zip")
for name in ("HMS", "RAMEDIS", "MME", "LIRICAL"):
    lines = archive.read(f"data/{name}.jsonl").decode().splitlines()
    print(f"{name:<8} {len(lines):>4} 个病人")
print("\nHMS 第 9 个病人，原样：")
print(archive.read("data/HMS.jsonl").decode().splitlines()[8])

### 结果怎么看
- 四组一共 1,122 个病人
- HMS 第 9 个病人的原样：一串 HPO 编号，诊断写了两个 OMIM 编号（`OMIM:106300`、`OMIM:183840`），`Department` 是空的。**一个病人列了两个诊断编号**，这一点第 3 步要处理

---

## 第 2 步 · 查重叠：哪几组真算"外部"

### 在干什么
把 RareBench 每一个病人，和 Phenopacket Store 0.1.27 里**同一个病**的病人逐个比症状，看是不是同一个人。

### 背景
"外部数据"最怕的是：看着是另一份数据，其实是同一批病人换了个包装。LIRICAL 组和 MME 组都是从**已发表的病例报告**里整理的，Phenopacket Store 也是；同一篇病例报告，完全可能被两边都收进去。要是这样，用它考就不是"外部验证"，而且：

- 同一个病人如果在我们的**开发集**里，他就在"找相似病人"的病例库里，等于把答案放在了工具里
- RareBench 不写出处论文，我们**没法按论文遮挡**

### 用了什么技术，为什么
- 比法：两个病人的症状集合的重合程度（**Jaccard 相似度**：共有的症状数 ÷ 两人症状合起来的总数）。1 表示一模一样，0.8 以上基本可以认为是同一个人
- 先把编号换成同一版 HPO 的写法再比，免得同一个症状因为写法不同被当成两个

### 公司里什么情况怎么做
- **合并或引入外部数据集之前，先做去重**（deduplication）。机器学习里"训练集和测试集里有同一条数据"是最常见的评测事故之一，大模型评测里专门叫它"数据污染"（contamination）
- **完全一样**好查；**几乎一样**（同一个病人被两个团队各整理了一遍，编号略有出入）才是难点，所以要用相似度，不能只比是否相等

### 执行
下面在本机算，不花钱，约半分钟：

In [ ]:
from collections import Counter, defaultdict
from raredx.data import cases as store
from raredx.knowledge import hpo
knowledge = hpo.load(ROOT / "data/raw/hpo/v2026-09-01")
release = store.read(ROOT / "data/processed/phenopacket-store/0.1.27")
rarebench = store.read(ROOT / "data/processed/rarebench/6f054e0")
same_disease = defaultdict(list)
for c in release:
    same_disease[c.disease_id].append(c)

def closest(patient):
    """同一个病（任何一个算对的 OMIM 编号）的已发表病人里，症状最像的那一个，和它的相似度"""
    mine = set(patient.hpo_ids)
    best, who = 0.0, None
    for disease in (patient.disease_id, *patient.also_accepted):
        for other in same_disease.get(disease, []):
            theirs = {knowledge.current(t) for t in other.hpo_ids} - {None}
            j = len(mine & theirs) / len(mine | theirs)
            if j > best:
                best, who = j, other
    return best, who

print(f"{'组':<8}{'人数':>5}{'没有同病病人':>10}{'相似度<0.5':>10}{'0.5–0.8':>9}{'≥0.8':>6}{'其中在开发集':>9}")
for name in ("HMS", "RAMEDIS", "MME", "LIRICAL"):
    group = [c for c in rarebench if c.cohort == name]
    found = [closest(c) for c in group]
    none = sum(who is None for _, who in found)
    low = sum(who is not None and j < 0.5 for j, who in found)
    mid = sum(0.5 <= j < 0.8 for j, _ in found)
    high = [who for j, who in found if j >= 0.8]
    print(f"{name:<8}{len(group):>5}{none:>12}{low:>12}{mid:>9}{len(high):>6}{sum(w.split == 'design' for w in high):>11}")

lirical_270 = next(c for c in rarebench if c.id == "RareBench_LIRICAL_270")
j, who = closest(lirical_270)
print(f"\n例子：RareBench_LIRICAL_270 和 {who.id}（{who.split}）相似度 {j:.2f}")
print("  诊断：", lirical_270.disease_label, "|", who.disease_label)
print("  症状：", "、".join(lirical_270.hpo_labels))

### 结果怎么看

| 组 | 人数 | 和 Phenopacket Store 的病人几乎一样（≥0.8） | 结论 |
|---|---|---|---|
| HMS | 88 | 0（只有 5 人找得到同病病人，相似度都不到 0.5） | **外部** |
| RAMEDIS | 624 | 0（95 人找得到同病病人，相似度都不到 0.5） | **外部** |
| MME | 40 | 0（它的 17 种病，0.1.27 里一个病人都没有） | 不用：出自病例报告，HPO 可能引用了这些报告，又没有出处没法遮 |
| LIRICAL | 370 | **22**（12 个在开发集，10 个在留出集），另有 105 个在 0.5–0.8 | **不用：和我们是同一批来源** |

**例子**：RareBench_LIRICAL_270 和留出集里的 PMID_28371217_proband，**同一个病、同样 9 个症状，相似度 1.00**，就是同一个人。它要是被当成外部题，第 21 章的留出集里已经考过了；更糟的是另外 12 个，它们就在"找相似病人"的病例库里。

所以外部验证只用 **HMS 和 RAMEDIS**。

> **这一步改变了原来的计划。** 项目第一天的总体计划（`docs/00_plan.md`）写的是"RareBench 公开子集：MME、HMS、LIRICAL、RAMEDIS"四组都用。查了重叠才知道 LIRICAL 组不能算外部。"别人都这么用"不等于"适合我们"。

### 面试怎么说
> "Before using RareBench as an external set I matched every patient against our own store by phenotype overlap: its LIRICAL subset holds 22 patients that are near-identical to ours — 12 of them sitting in the case bank our similar-patients tool searches — so only the clinic and registry subsets, HMS and RAMEDIS, count as external." 

---

## 第 3 步 · 读进来：诊断编号怎么认

### 在干什么
把 RareBench 的病人读成项目统一的"病例"格式，重点是**诊断编号怎么认**。

### 背景
我们所有系统都用 OMIM 编号回答，判分也只认 OMIM 编号（第 07 章）。RareBench 的诊断写成三种编号：OMIM、Orphanet（ORPHA，欧洲的罕见病目录）、中国罕见病目录（CCRD），而且常常一个病人列好几个：

| 病人 | RareBench 写的 | 是什么情况 |
|---|---|---|
| HMS 第 9 个 | `OMIM:106300`、`OMIM:183840` | 两个 OMIM 编号 |
| RAMEDIS 第 1 个 | `OMIM:251000`、`ORPHA:27`、`CCRD:71` | 同一个病的三种写法 |
| HMS 第 24 个 | `ORPHA:81`（抗合成酶综合征） | **一个 OMIM 编号都没有**：它不是单基因遗传病 |

### 用了什么技术，为什么
规则写在 `src/raredx/data/rarebench.py`，有测试，**在任何系统考这份题之前定下**：

1. 病人列出的**每一个 OMIM 编号都算对**（答中任意一个就行）
2. 列出的 Orphanet 编号，如果 Orphanet 自己把它**精确对应**（"E"类）到某个 OMIM 编号，那个也算对；只是"上下级"的对应不算（一个大类不等于其中某一个病）
3. 一个 OMIM 编号都没有的病人，诊断就记成它自己的编号（比如 `ORPHA:81`）。用 OMIM 回答的系统**不可能答对**，算错。真实门诊里也是这样：我们的系统只认识单基因病
4. 症状编号换成我们这一版 HPO 的写法（别名换成正名，作废的换成替代它的）

另外两件事：
- **没有出处论文，就没有东西要遮**。这些病人不来自论文，HPO 的资料不可能引用他们。考试照样做泄露审计（把工具的每一个回答都读一遍），只是没有"病人自己的论文"可找
- 判分程序改成"答中任意一个算对的编号都算对"（以前一个病人只有一个编号）

> **一个真实踩过的坑**：第一版的"诊断名称"只取第一个认得的 OMIM 编号的名字。HMS 第 6 个病人列了 8 个编号，前面是"非乳头状肾细胞癌"，后面是好几个系统性红斑狼疮。按编号判分没问题（任意一个都算对），但 DeepRare 的**宽松判分**是把诊断名称交给大模型裁判看的，裁判会以为确诊是"肾细胞癌"。改成把所有编号的病名都写上，用分号连起来。DeepRare 自己读 RareBench 也是这么做的。
>
> **第二个坑**：这些病人的"出处论文"一栏是空的，考卷文件每行末尾就多了一个制表符。提交代码时，自动检查里有一条"删掉行尾空白"，把它当成多余的空格删了，考卷就读不出来了。改法：让这条检查跳过 `.tsv` 数据文件（表格最后一列是空的，那是一个值，不是空白）。

### 公司里什么情况怎么做
- **两个系统的"标签体系"不一样时，对应规则要事先写死、写进代码、有测试**，而且宁严勿宽。不然考完一看分数低，很容易"再放宽一点"
- **答不了的题照样算在分母里**，单独报告有多少。去掉它们会让分数好看，但那不是真实情况

### 执行

In [ ]:
run("uv run raredx data build rarebench")
rarebench = store.read(ROOT / "data/processed/rarebench/6f054e0")
by_id = {c.id: c for c in rarebench}
for case_id in ("RareBench_HMS_009", "RareBench_RAMEDIS_001", "RareBench_HMS_024", "RareBench_HMS_006"):
    c = by_id[case_id]
    print(f"{case_id}: 判分认 {[c.disease_id, *c.also_accepted]}")
    print(f"    名称：{c.disease_label[:150]}")

### 结果怎么看
- HMS 第 9 个：两个 OMIM 编号都算对
- RAMEDIS 第 1 个：只有一个 OMIM 编号；`ORPHA:27` 精确对应的也是它，不重复
- HMS 第 24 个：判分认的是 `ORPHA:81`，没有任何系统能用 OMIM 编号答中，注定答不对
- HMS 第 6 个：列了 8 个编号，其中 7 个 OMIM 编号都算对；第 8 个 `ORPHA:93552`（儿童系统性红斑狼疮）在 Orphanet 里没有精确对应的 OMIM 编号，不添。名称把 8 个都写上了

---

## 第 4 步 · 选题、先写方案

### 在干什么
定下三份考卷，写好方案提交：[`docs/external_plan.md`](../external_plan.md)。

### 考卷

| 考卷 | 题数 | 给谁用 |
|---|---|---|
| `rarebench-hms` | 88 | 所有系统 |
| `rarebench-ramedis` | 624 | 不花钱的系统和用 DeepSeek 的系统 |
| `rarebench-ramedis-200` | 200（按固定种子从上一份里抽） | 按题收费的系统：三家通用大模型、DeepRare。和第 24 章的 `heldout-papers-200` 同一个道理：已经考过全部 624 道的系统，直接取这 200 道的答案，不重考 |

### 先算"地板"
看分数之前，先问一句：**什么都不会的系统能拿多少？**

RAMEDIS 的 624 个病人里，**苯丙酮尿症有 148 个**。一个永远只回答"苯丙酮尿症"的系统，第 1 名就有 **23.7%**。所以 RAMEDIS 上的分数要和这个地板比，不能单看。

### 主要结论（只有一个）

> **在两组外部病人上（HMS + RAMEDIS，共 712 道），最终 agent 的第 1 名命中率，是否高于固定流程。**

这是第 21 章的主要结论，换到别人整理的病人上再验一次。它问的是整个项目最核心的问题：**让大模型自己决定查什么，比把工具按固定顺序跑一遍再交给它，是不是真的更好**。两组合起来算，是因为 HMS 只有 88 道，单独测不出第 21 章那种大小的差别。

### 我的预期（写在考之前）
- **HMS 整体分数会低**：四分之一注定答不对，剩下的多是风湿免疫病，HPO 对它们的记录不如单基因病详细
- **HMS 上通用大模型可能追上甚至超过 agent**：这些病是教科书上的常见罕见病，大模型很熟
- **RAMEDIS 分数会高**：代谢病的化验异常很有特点，agent 第 1 名 45%–65%
- **主要结论成立，但领先幅度比第 21 章小**

### 一件要事先说的事：钱不够
三家通用大模型考 HMS 和 RAMEDIS-200 大约要 $6.5，OpenRouter 账户里只有 $5.10。方案里写好了顺序：先考 HMS，再考 RAMEDIS-200，钱不够就停在那里，没考的照实写"没考"，**不换成更便宜的模型**（换了就不是同一个比较了）。

### 公司里什么情况怎么做
- **报告任何分数之前，先报"地板"**：最常见类别的比例、随机猜的分数。地板很高的数据集上，一个看着不错的分数可能什么都不说明
- **预算不够时，事先定好先做什么、后做什么**，而不是做到一半临时挑

### 执行

In [ ]:
from raredx.evals import splits
from raredx.evals.hits import answerable
for name in ("rarebench-hms", "rarebench-ramedis", "rarebench-ramedis-200"):
    exam = [by_id[case_id] for case_id, _ in splits.read(ROOT / f"evals/splits/{name}.tsv")]
    top, count = Counter(c.disease_label for c in exam).most_common(1)[0]
    lost = sum(not answerable(c, knowledge) for c in exam)
    print(f"{name:<22} {len(exam):>4} 道  {len({c.disease_id for c in exam}):>3} 种病  注定答不对 {lost:>2} 道   "
          f"最常见：{top[:28]} {count} 道（地板 {count / len(exam):.1%}）")

### 结果怎么看
- HMS：88 道、36 种病，**24 道注定答不对**；最常见的病（肉芽肿性多血管炎）占 12.5%
- RAMEDIS：624 道、74 种病，没有注定答不对的；**苯丙酮尿症占 23.7%**
- RAMEDIS-200：200 道、40 种病，苯丙酮尿症占 23.5%，和全部 624 道差不多

### 面试怎么说
> "For the external test I fixed the reading rules before any run — every listed OMIM entry counts, an Orphanet code only through Orphanet's exact mapping, a patient with no OMIM entry stays in the denominator as a miss — and I reported the floor first: a quarter of RAMEDIS is phenylketonuria, so always answering PKU scores 23.7%." 

---

## 第 5 步 · 开考：每个系统只考一次

### 在干什么
冻结的 v1.0 和其他系统，在 HMS（88 道）和 RAMEDIS（624 道；按题收费的系统考其中固定的 200 道）上各考一次。

### 背景
做法和第 28 章第 5 步完全一样（`--final`、MLflow 留痕、泄露审计、补问）。这次照方案第 6 节处理了三件事：

| 出了什么事 | 怎么处理的 |
|---|---|
| LIRICAL 在 HMS 上第一次跑到一半，进程断了，记录不完整 | 按规则作废，重考一次（列表里没有分数的那一行就是作废的） |
| 接口故障：GPT-6.1 在 HMS 上 28 道没答上，Claude Sonnet 5.5 50 道 | 补问：GPT 补一轮补齐，Claude 补两轮补齐 |
| 考完 HMS，OpenRouter 的余额不够了 | 照方案停在那里：三家通用大模型**没考** RAMEDIS-200，没有换成更便宜的模型 |

### 用了什么技术，为什么
和第 28 章第 5 步一样，这里不重复。

### 公司里什么情况怎么做
- **预算用完时，按事先排好的顺序停下**，没考的照实写"没考"。临时换便宜的模型，就不是同一个比较了
- 作废、补问都按规则来，并且写进报告

### 执行

In [ ]:
run("uv run raredx eval runs --split rarebench-hms")
run("uv run raredx eval runs --split rarebench-ramedis")

### 结果怎么看
- HMS 上 `seeded-agent`（v1.0）第 1 名 **19.3%**，`llm-direct`（同一个模型直接猜）**31.8%**：agent 比直接猜低了一大截
- RAMEDIS 上 v1.0 **24.7%**，直接猜 **32.7%**：一样
- GPT-6.1、Claude Sonnet 5.5 有好几行，原因和第 28 章一样：分数高的是只补问几道的那几轮，合起来才是一次完整的考试
- 最上面几行 `hypothesis-agent`，代码版本后面带 `-dirty`（有还没提交的改动）：那是 v1.1 **开发时**在这里练习的记录，第 8 步讲

---

## 第 6 步 · 结果：主要结论不成立，还输给了直接猜

### 在干什么
回答方案里的问题，每一项都报，包括输掉的。

### 背景
主要结论是：**两组合起来（712 道），最终 agent 的第 1 名是否高于固定流程**。判断方法和第 28 章第 6 步一样：逐题配对、符号检验，p < 0.05 才算。

### 执行

In [ ]:
import csv
from raredx.evals.scoring import p_value, sign_test, wilson

def load(name):
    return list(csv.DictReader(open(ROOT / "docs" / name, encoding="utf-8"), delimiter="\t"))

def rate(rows, s, k=1):
    n = sum(r[f"{s}@{k}"] == "1" for r in rows)
    lo, hi = wilson(n, len(rows))
    return f"{n / len(rows):6.1%} [{lo:.1%}–{hi:.1%}]"

def paired(rows, a, b, k=1):
    w = sum(r[f"{a}@{k}"] == "1" and r[f"{b}@{k}"] == "0" for r in rows)
    l = sum(r[f"{b}@{k}"] == "1" and r[f"{a}@{k}"] == "0" for r in rows)
    return f"{w:>3} : {l:<3} {p_value(sign_test(w, l))}"

HMS, RAM = load("rarebench_hms_hits.tsv"), load("rarebench_ramedis_hits.tsv")
NAMES = {"phenotype-rank": "传统方法", "rank-then-check": "先排名再逐条核对", "similar-patients": "找相似病人",
         "lirical": "LIRICAL", "llm-direct": "大模型直接猜", "fixed-pipeline": "固定流程", "agent": "最终 agent（v1.0）"}

print("主要结论：两组合起来，第 1 名，只有 agent 对 : 只有固定流程对")
print("   ", paired(HMS + RAM, "agent", "fixed-pipeline"), f"（{len(HMS) + len(RAM)} 道）")
print("    前 10 名（不是主要结论）", paired(HMS + RAM, "agent", "fixed-pipeline", 10))

for label, rows in (("HMS", HMS), ("RAMEDIS", RAM)):
    print(f"\n{label}，{len(rows)} 道")
    print(f"{'':<20}{'第 1 名':<24}{'前 10 名'}")
    for s, name in NAMES.items():
        print(f"{name:<18}{rate(rows, s, 1):<24}{rate(rows, s, 10)}")
    print(f"    agent 对直接猜：第 1 名 {paired(rows, 'agent', 'llm-direct')}   前 10 名 {paired(rows, 'agent', 'llm-direct', 10)}")

print("\nHMS 上三家通用大模型直接猜：")
for s in ("gpt-6.1", "claude-sonnet-5.5", "gemini-4"):
    print(f"    {s:<20}{rate(HMS, s, 1)}")

### 结果怎么看
- **主要结论：54 : 45，p = 0.42，不成立。** 在外部病人上，没有证明 agent 胜过固定流程。前 10 名 agent 领先很多（96 : 7），但那不是事先定的主要结论，不能拿来替代它
- **更要紧的一条：agent 在两组上都显著输给了同一个模型直接猜**：HMS 4 : 15，RAMEDIS 47 : 97。前 10 名也输：HMS 50.0% 对 53.4%（6 : 9，不显著），RAMEDIS 69.7% 对 79.8%（31 : 94，显著）
- 它用的就是这个模型，却没用上这个模型知道的东西。这是 agent 的问题，下一步找原因
- HMS 上三家通用大模型（18.2%、22.7%、14.8%）都比 deepseek 直接猜（31.8%）低，和第 24 章留出集上的顺序正好反过来。RareBench 2024 年就连同答案公开了，很像是 deepseek 背过这份题。**但这不能给 agent 开脱**：agent 用的正是 deepseek，本可以占这个便宜
- 方案里的其他比较（DeepRare 两种判法、花费、用时）都在 [`docs/external_plan.md`](../external_plan.md) 第 8 节

> **一个真实踩过的坑**：RAMEDIS 的 148 个苯丙酮尿症病人，所有系统都几乎答不对。翻开题目：115 个写着"高酪氨酸血症"，可苯丙酮尿症病人的酪氨酸是**偏低**的；不少题同时写着"新生儿死亡""婴儿期死亡""儿童期死亡"。看起来是 RareBench 把原始化验记录转成 HPO 时出了错。按方案规矩，考完不改题、不改判分，照原样算，只在报告里写明。

### 面试怎么说
> "On two external sets the frozen agent did not beat the fixed pipeline — 54 to 45, p = 0.42 — and it lost to the same model asked directly: 4 to 15 on HMS, 47 to 97 on RAMEDIS. I reported that as the result, then went looking for why."

---

## 第 7 步 · 为什么输：答案找到了，却没排第一

### 在干什么
读错题、算数，找出输的原因。**这一步是考完以后才做的，不是事先定的分析**，所以它只能指方向，不能当证据。

### 背景
从三个角度看：
1. **和内部验证比，掉了多少**：同一个系统，在留出集和两组外部病人上的分数并排
2. **"找到"还是"选对"**：几路名单合起来，正确答案在不在里面？在的话，有没有排到第一？
3. **翻原始数据**：知识库里，这些病到底记了多少条症状？

### 用了什么技术，为什么
第 2 个角度在工业界叫**上限分析**（oracle analysis），专门用在"先找候选、再排序"的两段式系统上（搜索、推荐都是这个结构）：
- 正确答案根本不在候选名单里 → 是**"找"**的问题，要加来源、扩名单
- 在名单里，却没排到第一 → 是**"选"**的问题，要改排序和判断

**打个比方**：招聘时最后没招到合适的人，是简历池里根本没有合适的人，还是有、但面试时选错了？两种情况的改法完全不同。

### 公司里什么情况怎么做
- 输了以后**先拆原因、再动手改**，不要凭感觉加功能
- **按数据来源切开看**（slice analysis）：一个平均分会把"某一类数据上全军覆没"藏起来

### 执行

In [ ]:
H = load("comparison_hits.tsv")
print("1. 第 1 名，同一个系统在三份题上")
print(f"{'':<20}{'留出集':>8}{'HMS':>8}{'RAMEDIS':>9}")
for s in ("llm-direct", "phenotype-rank", "lirical", "rank-then-check", "fixed-pipeline", "agent"):
    f = lambda rows: f"{sum(r[s + '@1'] == '1' for r in rows) / len(rows):.1%}"
    print(f"{NAMES[s]:<18}{f(H):>8}{f(HMS):>8}{f(RAM):>9}")

print("\n2. 找到，还是选对？")
print(f"{'':<10}{'几路名单合起来的前 10 名里有正确答案':>24}{'v1.0 第 1 名':>12}")
for label, rows, lists in (("留出集", H, ("llm-direct", "rank-then-check", "phenotype-rank", "lirical")),
                           ("HMS", HMS, ("llm-direct", "rank-then-check", "phenotype-rank", "lirical", "similar-patients")),
                           ("RAMEDIS", RAM, ("llm-direct", "rank-then-check", "phenotype-rank", "lirical", "similar-patients"))):
    pool = sum(any(r[f"{s}@10"] == "1" for s in lists) for r in rows) / len(rows)
    print(f"{label:<10}{pool:>24.1%}{sum(r['agent@1'] == '1' for r in rows) / len(rows):>20.1%}")

print("\n   直接猜第 1 名对、agent 第 1 名错的题里，agent 自己的前 10 名里其实有正确答案的：")
for label, rows in (("HMS", HMS), ("RAMEDIS", RAM)):
    lost = [r for r in rows if r["llm-direct@1"] == "1" and r["agent@1"] == "0"]
    print(f"   {label:<8}{sum(r['agent@10'] == '1' for r in lost)} / {len(lost)}")

第 3 个角度，翻原始数据。HPO 的疾病标注文件（`phenotype.hpoa`）每一行是"某个病有某个症状"。数一数几个病各有多少行：

In [ ]:
from collections import Counter
rows = (line.split("\t") for line in open(ROOT / "data/raw/hpo/v2026-09-01/phenotype.hpoa", encoding="utf-8") if not line.startswith("#"))
count, names = Counter(), {}
for r in rows:
    if r[0] == "database_id" or r[2] == "NOT" or r[10] != "P":   # 表头；"没有这个症状"；不是症状的行（遗传方式等）
        continue
    count[r[0]] += 1
    names[r[0]] = r[1]
print(f"{'':<44}{'OMIM 的标注':>10}{'Orphanet 的标注':>16}")
for omim, orpha in (("OMIM:109650", "ORPHA:117"), ("OMIM:152700", "ORPHA:536"), ("OMIM:608710", "ORPHA:900"), (None, "ORPHA:81")):
    print(f"{names[orpha]:<42}{(str(count[omim]) + ' 条') if omim else '没有':>10}{count[orpha]:>14} 条")
diseases = {d.split(":")[0] for d in count}
print("\n整个文件：", {db: sum(1 for d in count if d.startswith(db + ":")) for db in sorted(diseases)}, "种病")

### 结果怎么看
**1. 掉了多少**：所有靠 HPO 标注的系统都掉了很多（v1.0 从 42.4% 掉到 19.3% 和 24.7%），只有直接猜反而涨了（14.3% → 31.8%、32.7%）。

**2. 找到还是选对**：
- 几路名单合起来，正确答案在前 10 名里的比例（留出集约 62%、HMS 约 62%、RAMEDIS 约 86%），**远高于** v1.0 的第 1 名。差的那一截，是"答案明明在名单里，却没排到第一"
- 直接猜对、agent 错的题里，**绝大多数 agent 自己的前 10 名里就有正确答案**：HMS 15 道里有 12 道，RAMEDIS 97 道里有 85 道。它找到了，最后却选了别的
- 所以主要是**"选"**的问题，不是"找"的问题

**3. 原始数据**：
- 白塞病（一种反复口腔溃疡、眼睛发炎的免疫病）在 OMIM 那边只记了 15 条症状，Orphanet 那边记了 85 条；红斑狼疮 21 对 44；肉芽肿性多血管炎 32 对 80
- 抗合成酶综合征（HMS 里"注定答不对"的病之一）只在 Orphanet 那边有
- 而我们的知识库**只读了 OMIM 那一部分**（[`src/raredx/knowledge/hpo.py`](../../src/raredx/knowledge/hpo.py) 读文件时只留 `OMIM:` 开头的行），Orphanet 的 4,000 多种病全丢掉了

**合在一起，v1.0 输的原因**：
- 工具和提示词衡量的都是"一个病能解释病人多少症状"，没有一样在衡量"这个病常不常见"
- 相对常见的病在我们的知识库里记得少，极罕见的单基因病记得全。于是 agent 常把常见病换成一个症状更"对得上"的罕见模仿病，比如把白塞病换成 ADA2 缺乏症
- 提示词又把模型自己的知识放在次要位置：模型知道答案，也只能在工具的名单之外"补充"

### 面试怎么说
> "An oracle analysis showed it was a ranking failure, not a retrieval one: in most cases the model got right and the agent missed, the agent had the answer in its own top ten. Every tool scored how well a disease explains the findings and none how common it is, while the knowledge base — which kept only OMIM rows — annotates common rare diseases far more thinly than ultra-rare ones."

---

## 第 8 步 · 输了以后：v1.1（hypothesis agent）

### 在干什么
按第 7 步找到的原因改 agent，然后**换一份没碰过的题**来考。

### 背景：读过错题的考卷，就不能再当考卷了
第 7 步我把 HMS 和 RAMEDIS 的错题一道道读过了。照着错题改出来的系统，再在同一份题上考，分数一定好看，但说明不了什么。所以这两组**从此改作练习卷**，代码里专门列在 `splits.PRACTICE`，之后在上面跑的都只是练习成绩，系统名后面带 `-practice`。

**打个比方**：模拟考的卷子你已经对过答案了，再考一遍得高分，不代表期末能考好。

### v1.1 改了什么（代码 `788e56e`，系统名 `hypothesis-agent`，提示词 `hypotheses-v2`）
1. **先让模型不看工具**，凭自己的知识列出怀疑的病（就是"直接猜"那个问题）
2. 每个病在知识库里找到、逐条核对症状。**用模型写的编号**；知识库不认识那个编号时，才按名字找
3. 工具的前 10 名作为第二份名单，**两份名单一起**交给 agent，提示词让它"两边同等看待"
4. 每个候选标上 **Orphanet 的患病率等级**；提示词说"吻合程度差不多时，常见的更可能；患病率未知不等于罕见"
5. 打开查疾病介绍的工具；答案不足 10 个时，两份名单按排名合并补齐

> **一个真实踩过的坑**：第 2 条一开始是"按病名重新找编号"，想纠正模型写错的编号。拿已经记下的直接猜答案一核对，按名字找回来的，在每个门槛上都是**丢的比找回的多**：模型写的编号其实比按名字找更可靠。所以改成了"先信模型的编号"。我原先以为"直接猜输是因为编号写错"，这个想法是错的。

### 新的外部考卷：MME
RareBench 的第三组 **MME**（40 道，17 种病）。第 2 步原本没用它：它出自已发表的病例报告，HPO 的资料可能就引用了这些报告，又不写出处，没法遮挡。现在选它，是因为改版之前**没有任何系统考过它**，而且它的 17 种病在 Phenopacket Store 里一个都没有。那个缺点还在，所以**凡是用 HPO 的系统，在它上面的分数都可能偏高**；题也少，误差约正负 15 个百分点，其中 11 道是同一种病。

v1.0 也在 MME 上考一次（第一次只记下 40 道里的 11 道，很可能因为另一场考试同时在写 MLflow，按规则作废重考）。

### 用了什么技术，为什么
- **练习集 / 测试集分开**：机器学习最基本的规矩。看过的、调过的，一律算练习
- **内部回归检查**：在第 24 章固定的 200 道留出题上也考一次，确认改版没有把原来的本事改坏（工业界叫回归测试，regression test）

### 公司里什么情况怎么做
- 测试集一旦被用来找原因、调系统，就**降级为开发集**，并且换一份新的测试集。很多团队会给测试集记"被看过几次"
- 改版以后，**新数据上的成绩**和**老数据上的回归检查**都要报

### 执行

In [ ]:
print("练习卷（对着它们改过，只说明方向，不是考试成绩），第 1 名：")
for label, rows in (("HMS", HMS), ("RAMEDIS", RAM)):
    print(f"  {label:<8} v1.1 {rate(rows, 'hypothesis-agent-practice')}   v1.0 {rate(rows, 'agent')}   直接猜 {rate(rows, 'llm-direct')}"
          f"   v1.1 对直接猜 {paired(rows, 'hypothesis-agent-practice', 'llm-direct')}")

M = load("rarebench_mme_hits.tsv")
print(f"\n新的外部考卷 MME，{len(M)} 道，每个系统只考一次：")
print(f"{'':<20}{'第 1 名':<24}{'前 10 名':<14}{'v1.1 逐题（第 1 名）'}")
for s, name in (("hypothesis-agent", "v1.1"), ("agent", "v1.0"), ("fixed-pipeline", "固定流程"),
                ("rank-then-check", "先排名再逐条核对"), ("lirical", "LIRICAL"), ("llm-direct", "大模型直接猜")):
    vs = "" if s == "hypothesis-agent" else paired(M, "hypothesis-agent", s)
    print(f"{name:<18}{rate(M, s, 1):<24}{rate(M, s, 10):<14}{vs}")

R = load("heldout200_hypothesis_hits.tsv")
print(f"\n回归检查：第 24 章固定的 {len(R)} 道留出题，第 1 名：")
for s, name in (("hypothesis-agent", "v1.1"), ("agent", "v1.0"), ("fixed-pipeline", "固定流程"), ("llm-direct", "大模型直接猜")):
    vs = "" if s == "hypothesis-agent" else paired(R, "hypothesis-agent", s)
    print(f"  {name:<14}{rate(R, s, 1):<24}{vs}")

### 结果怎么看
- **练习卷**：HMS 上 v1.1 37.5%，比直接猜（31.8%）高；RAMEDIS 上 30.6%，还是比直接猜（32.7%）低。都对着它们改过，只说明方向
- **MME（干净的新外部卷）**：v1.1 72.5%，**比 v1.0 的 80.0% 少对 3 道**（逐题 1 : 4，不显著）。MME 和 HMS 正好相反：病极罕见，模型自己几乎不认识（直接猜只对 2 道），靠知识库的工具很强。v1.1 丢的 4 道，正确答案都在工具名单的前三名里；它不再那么信工具的第一名，又去读了疾病介绍，换成了另一个同样罕见的病。这就是这次修改的代价：**模型占优的题上多对了很多，工具占优的题上少对了几道**
- **回归检查**：留出 200 道上 v1.1 45.0%、v1.0 43.0%（12 : 8，不显著）；对固定流程 19 : 5。没有退步

**所以，老实说**：v1.1 修好了"模型知道的病被工具压下去"，但在**能算数的外部卷**上，还没有证明它比 v1.0 好。

### 面试怎么说
> "Once I had read HMS and RAMEDIS case by case they became practice sets. The fix — the model's own hypotheses first, checked beside the tools', with prevalence shown — helped there, but on a fresh external set, MME, it scored 72.5% against v1.0's 80%, 1 to 4: it now trusts the tools less, which costs it on ultra-rare diseases the model doesn't know. No regression on the held-out 200. So v1.1 is not yet shown better than v1.0 on clean external data."

---

## 本章小结

| 步骤 | 用的技术 | 在公司里的作用 |
|---|---|---|
| 1 找外部数据 | 固定在 HuggingFace 的一次提交 | 外部数据也要固定版本 |
| 2 查重叠 | 逐个比对症状 | 确认"外部"真的是外部 |
| 3 读进来 | 编号对应规则写进代码、有测试 | 标签体系不同时，规则事先写死 |
| 4 选题、先写方案 | 预注册；先算"地板" | 分数先和"什么都不会"比 |
| 5 开考 | `--final`、作废重考、补问、预算用完按顺序停 | 正式评测按规则处理意外 |
| 6 结果 | 逐题配对、符号检验 | 输的结论照实报 |
| 7 为什么输 | 上限分析、按来源切开看、翻原始数据 | 先拆原因再动手 |
| 8 v1.1 | 测试集降级为练习集、新的外部卷、回归检查 | 改版后的成绩怎么报才可信 |

**一句话**：换了一批别人整理的病人，v1.0 没站住，还输给了同一个模型直接猜；原因主要是"找到了答案却没排第一"，以及知识库只读了 OMIM、只算"像不像"不算"常不常见"。v1.1 部分修好了，但在干净的新外部卷上还没证明比 v1.0 好。

## 面试题

> **Q: Your agent lost to the bare model on external data. What did you do?**
>
> A: "Reported it first, as the pre-registered result. Then an oracle analysis: the answer was usually in the agent's own candidate list, so it was a ranking failure — the tools measured fit but not prevalence, and the knowledge base had dropped every Orphanet disease. I fixed part of that, demoted the two sets I had read to practice, and tested on a fresh set, where the new version was not better. That is why the next version changes the architecture and how development data is split, not just the prompt."

> **Q: Why can't you keep using HMS as a test set after the fix?**
>
> A: "Because I read its errors to design the fix. A set you've tuned against measures how well you fit it, not how well you generalise — so it becomes development data and a new set takes its place."

## 下一章

**30 · v2.0 第一步：先离线试一试"校准打分"**。第 7 步说明主要是"选"的问题。下一版的计划（[`docs/roadmap.md`](../roadmap.md)）是把"怎么在候选里选"从模型凭感觉，改成用多个来源的数据校准过的打分。动手改系统之前，先用已经记下的答案离线算一算：这条路通不通，一分钱不花。